# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [5]:
# Install necessary libraries: smolagents for agent logic and wikipedia for additional search capabilities
!pip install -q smolagents[transformers] wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.5/161.5 kB 15.1 MB/s eta 0:00:00


## 1) Define KB

In [10]:
# Define the knowledge base snippets that the agent will use for information lookup
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries defined:', len(kb_snippets))

KB entries defined: 5


## 2) Define tools

In [11]:
from smolagents import Tool, TransformersModel, ToolCallingAgent

class KBLookupTool(Tool):
    """Tool to search through a local knowledge base of strings."""
    name = "kb_lookup"
    description = "Looks up relevant information from a custom knowledge base snippets."
    inputs = {"query": {"type": "string", "description": "The search query to look for in the knowledge base."}}
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        # Convert query to lowercase for case-insensitive matching
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in q.split())
        ]
        return "".join(matches) if matches else "No KB match."


class MathTool(Tool):
    """Tool for basic arithmetic operations."""
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {
        "a": {"type": "number", "description": "The first number."},
        "b": {"type": "number", "description": "The second number."},
        "op": {"type": "string", "description": "The operation to perform: 'add' or 'multiply'.", "nullable": True}
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        # Perform multiplication if requested, otherwise default to addition
        if op == "multiply":
            return str(a * b)
        return str(a + b)

# Instantiate the custom tools with the provided knowledge base
kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

## 3) Model (tiny local)

In [7]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Initialize the local LLM using the Transformers library
# We use 'auto' for device_map to leverage GPU if available
model = TransformersModel(
    model_id=MODEL_ID,
    device_map="auto",
    max_new_tokens=200
)

print("Model ready:", MODEL_ID)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model ready: TinyLlama/TinyLlama-1.1B-Chat-v1.0


## 4) Agent

In [12]:
# Create the agent instance and equip it with our tools
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool], # The tools we defined earlier
    model=model,
    max_steps=3, # Maximum reasoning steps to prevent infinite loops
    instructions=(
        "You are a helpful assistant that uses a local knowledge base and a math tool to answer questions. "
        "Always check the knowledge base for definitions and use the math tool for calculations."
    ),
)

print("Agent initialized.")

Agent initialized.


## 5) Test queries

In [13]:
# Define a set of test queries to verify the agent's functionality
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("\n---")
    print("Q:", q)
    # Run the agent for each query and display the final answer
    result = agent.run(q)
    print("Answer:", result)


---
Q: Add 12 and 30.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Add 12 and 30.                                                                                                  │
│                                                                                                                 │
╰─ TransformersModel - TinyLlama/TinyLlama-1.1B-Chat-v1.0 ────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'math_tool' with arguments: {'a': {'type': 'number', 'description': 'The first number'}, 'b':     │
│ {'type': 'number', 'description': 'The second number'}, 'op': {'type': 'string', 'description': "The operation  │
│ to perform: 'add' or 'multiply'.", 'nullable': True}}                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Step 3: Duration 8.88 seconds| Input tokens: 4,498 | Output tokens: 534]

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Reached max steps.

[Step 4: Duration 4.16 seconds| Input tokens: 5,832 | Output tokens: 617]

Answer: An agentic AI loop is a type of AI agent that is capable of performing a specific task repeatedly without being programmed to do so. It is a type of AI loop that is designed to perform a task repeatedly, without any human intervention. Agentic AI loops are often used in industries such as healthcare, finance, and education, where repetitive tasks are common.
